In [1]:
import os
import re
import pandas as pd

PLOTS_DIR = "plots"

# Specify which subdirectories to summarize
SUMMARY_DIRS = [
    "flow_NCSF_cond4_global",
    "flow_NCSF_cond4_local",
    # "flow_NSF_cond4_global",
    # "flow_NSF_cond4_local",
    "diff_large_cond4_global",
    "diff_large_cond4_local",
]


In [2]:
def parse_bdt_results(filepath):
    result = {"per_feature_auc": {}, "feature_importances": {},
              "full_auc": None, "full_auc_std": None}
    section = None
    with open(filepath) as f:
        for line in f:
            line = line.rstrip()
            if "Per-feature BDT performance" in line:
                section = "per_feature"
            elif "Feature importances (full BDT)" in line:
                section = "importances"
            elif "Pairwise BDT performance" in line:
                section = "pairwise"
            elif section == "per_feature":
                m = re.match(r"Feature (.+?) BDT AUC: ([0-9]+(?:\.[0-9]+)?) . ([0-9]+(?:\.[0-9]+)?)", line)
                if m:
                    result["per_feature_auc"][m.group(1)] = (float(m.group(2)), float(m.group(3)))
            elif section == "importances":
                m = re.match(r"(.+?): ([0-9]+(?:\.[0-9]+)?) . ([0-9]+(?:\.[0-9]+)?)", line)
                if m:
                    result["feature_importances"][m.group(1)] = (float(m.group(2)), float(m.group(3)))
                m2 = re.search(r"auc ([0-9]+(?:\.[0-9]+)?) \\pm ([0-9]+(?:\.[0-9]+)?)", line)
                if m2:
                    result["full_auc"] = float(m2.group(1))
                    result["full_auc_std"] = float(m2.group(2))
    return result


def summarize_directory(subdir):
    dirpath = os.path.join(PLOTS_DIR, subdir)
    COLLECTION_ORDER = [
        "OuterTrackerBarrelCollection", "OuterTrackerEndcapCollection",
        "InnerTrackerBarrelCollection", "InnerTrackerEndcapCollection",
        "VertexBarrelCollection", "VertexEndcapCollection",
    ]
    present = {f.replace("bdt_results_", "").replace(".txt", "") for f in os.listdir(dirpath)
               if f.startswith("bdt_results_") and f.endswith(".txt")}
    files = [f"bdt_results_{c}.txt" for c in COLLECTION_ORDER if c in present]
    rows = []
    for fname in files:
        collection = fname.replace("bdt_results_", "").replace(".txt", "")
        parsed = parse_bdt_results(os.path.join(dirpath, fname))
        row = {"collection": collection}
        if parsed["full_auc"] is not None:
            row["full_auc"] = f'{parsed["full_auc"]:.4f} ± {parsed["full_auc_std"]:.4f}'
        else:
            row["full_auc"] = "—"
        for feat, (mean, std) in parsed["per_feature_auc"].items():
            row[f"auc_{feat}"] = f"{mean:.4f} ± {std:.4f}"
        for feat, (mean, std) in parsed["feature_importances"].items():
            row[f"imp_{feat}"] = f"{mean:.4f} ± {std:.4f}"
        rows.append(row)
    return pd.DataFrame(rows)


In [3]:
for subdir in SUMMARY_DIRS:
    dirpath = os.path.join(PLOTS_DIR, subdir)
    if not os.path.isdir(dirpath):
        print(f"Skipping {subdir}: directory not found")
        continue
    files = [f for f in os.listdir(dirpath) if f.startswith("bdt_results_") and f.endswith(".txt")]
    if not files:
        print(f"Skipping {subdir}: no bdt_results_*.txt files found")
        continue

    df = summarize_directory(subdir)

    print(f"=== {subdir} ===")

    # Full BDT AUC table
    print("-- Full BDT AUC --")
    display(df[["collection", "full_auc"]].set_index("collection"))

    # # Per-feature AUC table
    # auc_cols = [c for c in df.columns if c.startswith("auc_")]
    # if auc_cols:
    #     print("-- Per-feature BDT AUC --")
    #     auc_df = df[["collection"] + auc_cols].set_index("collection")
    #     auc_df.columns = [c.replace("auc_", "") for c in auc_df.columns]
    #     display(auc_df)

    # Feature importances table
    imp_cols = [c for c in df.columns if c.startswith("imp_")]
    if imp_cols:
        print("-- Feature importances (full BDT) --")
        imp_df = df[["collection"] + imp_cols].set_index("collection")
        imp_df.columns = [c.replace("imp_", "") for c in imp_df.columns]
        display(imp_df)


=== flow_NCSF_cond4_global ===
-- Full BDT AUC --


,full_auc
collection,
OuterTrackerBarrelCollection,0.6314 ± 6.8632
OuterTrackerEndcapCollection,0.5350 ± 0.0013
InnerTrackerBarrelCollection,0.5846 ± 9.9467
InnerTrackerEndcapCollection,0.5361 ± 0.0005
VertexBarrelCollection,0.5455 ± 3.9991
VertexEndcapCollection,0.5403 ± 0.0002


-- Feature importances (full BDT) --


,log($E$) [Gev],$t$ [s],$r$ [mm],$\phi$,$z$ [mm]
collection,,,,,
OuterTrackerBarrelCollection,0.0134 ± 0.0007,0.0214 ± 0.0009,0.8349 ± 0.0016,0.0323 ± 0.0006,0.0979 ± 0.0014
OuterTrackerEndcapCollection,0.2489 ± 0.0123,0.1586 ± 0.0090,0.1623 ± 0.0032,0.0970 ± 0.0033,0.3332 ± 0.0098
InnerTrackerBarrelCollection,0.0403 ± 0.0012,0.0872 ± 0.0045,0.5970 ± 0.0048,0.1280 ± 0.0016,0.1474 ± 0.0014
InnerTrackerEndcapCollection,0.1338 ± 0.0072,0.1926 ± 0.0028,0.2668 ± 0.0019,0.1301 ± 0.0008,0.2767 ± 0.0082
VertexBarrelCollection,0.0939 ± 0.0017,0.0769 ± 0.0007,0.6622 ± 0.0026,0.0926 ± 0.0010,0.0745 ± 0.0008
VertexEndcapCollection,0.1830 ± 0.0133,0.0977 ± 0.0031,0.1584 ± 0.0019,0.1407 ± 0.0028,0.4202 ± 0.0129


=== flow_NCSF_cond4_local ===
-- Full BDT AUC --


...

,log($E$) [Gev],$t$ [s],$r$ [mm],$\phi$,$z$ [mm]
collection,,,,,
OuterTrackerBarrelCollection,0.1548 ± 0.0064,0.1496 ± 0.0023,0.3608 ± 0.0153,0.1546 ± 0.0045,0.1803 ± 0.0088
OuterTrackerEndcapCollection,0.2259 ± 0.0080,0.1609 ± 0.0022,0.1504 ± 0.0023,0.1166 ± 0.0017,0.3462 ± 0.0100
InnerTrackerBarrelCollection,0.0443 ± 0.0008,0.0730 ± 0.0015,0.5869 ± 0.0018,0.1593 ± 0.0020,0.1365 ± 0.0024
InnerTrackerEndcapCollection,0.1394 ± 0.0046,0.2753 ± 0.0025,0.2075 ± 0.0028,0.1198 ± 0.0023,0.2580 ± 0.0063
VertexBarrelCollection,0.0805 ± 0.0064,0.1295 ± 0.0082,0.4456 ± 0.0114,0.1593 ± 0.0076,0.1850 ± 0.0108
VertexEndcapCollection,0.1711 ± 0.0123,0.1141 ± 0.0030,0.1649 ± 0.0044,0.1632 ± 0.0027,0.3868 ± 0.0134


=== diff_large_cond4_local ===
-- Full BDT AUC --


,full_auc
collection,
OuterTrackerBarrelCollection,0.5528 ± 0.0001
OuterTrackerEndcapCollection,0.5281 ± 0.0015
InnerTrackerBarrelCollection,0.5267 ± 0.0001
InnerTrackerEndcapCollection,0.5315 ± 0.0006
VertexBarrelCollection,0.5124 ± 0.0002
VertexEndcapCollection,0.5396 ± 0.0004


-- Feature importances (full BDT) --


,log($E$) [Gev],$t$ [s],$r$ [mm],$\phi$,$z$ [mm]
collection,,,,,
OuterTrackerBarrelCollection,0.0910 ± 0.0010,0.1439 ± 0.0011,0.2493 ± 0.0042,0.0628 ± 0.0015,0.4530 ± 0.0053
OuterTrackerEndcapCollection,0.2344 ± 0.0168,0.1626 ± 0.0030,0.1651 ± 0.0074,0.0837 ± 0.0033,0.3542 ± 0.0149
InnerTrackerBarrelCollection,0.0864 ± 0.0032,0.1723 ± 0.0018,0.2430 ± 0.0023,0.0807 ± 0.0019,0.4175 ± 0.0044
InnerTrackerEndcapCollection,0.1321 ± 0.0069,0.2981 ± 0.0031,0.1930 ± 0.0024,0.0928 ± 0.0037,0.2840 ± 0.0067
VertexBarrelCollection,0.1298 ± 0.0041,0.2045 ± 0.0038,0.2064 ± 0.0043,0.1107 ± 0.0068,0.3485 ± 0.0127
VertexEndcapCollection,0.1911 ± 0.0251,0.1205 ± 0.0025,0.1727 ± 0.0066,0.1075 ± 0.0029,0.4082 ± 0.0175


In [4]:
from collections import defaultdict

def _get_prefix(dirname):
    if dirname.endswith("_global"):
        return dirname[:-len("_global")]
    if dirname.endswith("_local"):
        return dirname[:-len("_local")]
    return dirname

# Group SUMMARY_DIRS into (global, local) pairs by common prefix
_groups = defaultdict(lambda: {"global": None, "local": None})
for _d in SUMMARY_DIRS:
    _prefix = _get_prefix(_d)
    if _d.endswith("_global"):
        _groups[_prefix]["global"] = _d
    elif _d.endswith("_local"):
        _groups[_prefix]["local"] = _d

COLLECTION_DISPLAY = {
    "OuterTrackerBarrelCollection": "Outer Tracker Barrel",
    "InnerTrackerBarrelCollection": "Inner Tracker Barrel",
    "VertexBarrelCollection": "Vertex Barrel",
    "OuterTrackerEndcapCollection": "Outer Tracker Endcap",
    "InnerTrackerEndcapCollection": "Inner Tracker Endcap",
    "VertexEndcapCollection": "Vertex Endcap",
}

LATEX_COLLECTION_ORDER = [
    "OuterTrackerBarrelCollection",
    "InnerTrackerBarrelCollection",
    "VertexBarrelCollection",
    "OuterTrackerEndcapCollection",
    "InnerTrackerEndcapCollection",
    "VertexEndcapCollection",
]

def _get_parsed(dirpath, collection):
    if dirpath is None:
        return None
    filepath = os.path.join(PLOTS_DIR, dirpath, f"bdt_results_{collection}.txt")
    if not os.path.exists(filepath):
        return None
    return parse_bdt_results(filepath)

def _discover_feature_keys(global_dir, local_dir):
    for col in LATEX_COLLECTION_ORDER:
        for d in [global_dir, local_dir]:
            p = _get_parsed(d, col)
            if p and p["feature_importances"]:
                return list(p["feature_importances"].keys())
    return []

def _build_row(display_name, g_parsed, l_parsed, feature_keys):
    def fmt(x):
        return f"{x:.3f}"

    def auc_str(parsed):
        if parsed and parsed["full_auc"] is not None:
            return fmt(parsed["full_auc"])
        return "{-}"

    def imp_str(parsed, key):
        if parsed and key in parsed["feature_importances"]:
            return fmt(parsed["feature_importances"][key][0])
        return "{-}"

    g_auc = auc_str(g_parsed)
    l_auc = auc_str(l_parsed)
    g_imps = [imp_str(g_parsed, k) for k in feature_keys]
    l_imps = [imp_str(l_parsed, k) for k in feature_keys]

    cells = [display_name, g_auc] + g_imps + [l_auc] + l_imps
    return "    " + " & ".join(cells) + r" \\"

def make_latex_table(global_dir, local_dir):
    feature_keys = _discover_feature_keys(global_dir, local_dir)

    left_label  = (global_dir  if global_dir  else "—").replace("_", r"\_")
    right_label = (local_dir   if local_dir   else "—").replace("_", r"\_")

    lines = [
        r"\begin{tabular}{|c|c|ccccc|c|ccccc|}",
        r"    \hline",
        r"    \multirow{2}{*}{Collection Name} &",
        f"    \\multicolumn{{6}}{{c|}}{{{left_label}}} &",
        f"    \\multicolumn{{6}}{{c|}}{{{right_label}}} \\\\",
        r"    \cline{2-13}",
        r"    & ROC AUC &",
        r"    \multicolumn{5}{c|}{BDT Feature Importances} &",
        r"    ROC AUC &",
        r"    \multicolumn{5}{c|}{BDT Feature Importances} \\",
        r"    \cline{3-7} \cline{9-13}",
        r"    & & $\log(E)$ & $t$ & $r$ & $\phi$ & $z$",
        r"      & & $\log(E)$ & $t$ & $r$ & $\phi$ & $z$ \\",
        r"    \hline",
    ]

    for collection in LATEX_COLLECTION_ORDER:
        g_data = _get_parsed(global_dir, collection)
        l_data = _get_parsed(local_dir, collection)
        display_name = COLLECTION_DISPLAY.get(collection, collection)
        lines.append(_build_row(display_name, g_data, l_data, feature_keys))
        lines.append(r"    \hline")

    # "All collections" row — looks for bdt_results_all_collections.txt
    g_all = _get_parsed(global_dir, "all_collections")
    l_all = _get_parsed(local_dir, "all_collections")
    lines.append(_build_row("All collections", g_all, l_all, feature_keys))
    lines.append(r"    \hline")
    lines.append(r"    \end{tabular}")

    return "\n".join(lines)

for _prefix, _dirs in sorted(_groups.items()):
    print(make_latex_table(_dirs["global"], _dirs["local"]))
    print()


\begin{tabular}{|c|c|ccccc|c|ccccc|}
    \hline
    \multirow{2}{*}{Collection Name} &
    \multicolumn{6}{c|}{diff\_large\_cond4\_global} &
    \multicolumn{6}{c|}{diff\_large\_cond4\_local} \\
    \cline{2-13}
    & ROC AUC &
    \multicolumn{5}{c|}{BDT Feature Importances} &
    ROC AUC &
    \multicolumn{5}{c|}{BDT Feature Importances} \\
    \cline{3-7} \cline{9-13}
    & & $\log(E)$ & $t$ & $r$ & $\phi$ & $z$
      & & $\log(E)$ & $t$ & $r$ & $\phi$ & $z$ \\
    \hline
    Outer Tracker Barrel & 0.576 & 0.155 & 0.150 & 0.361 & 0.155 & 0.180 & 0.553 & 0.091 & 0.144 & 0.249 & 0.063 & 0.453 \\
    \hline
    Inner Tracker Barrel & 0.581 & 0.044 & 0.073 & 0.587 & 0.159 & 0.137 & 0.527 & 0.086 & 0.172 & 0.243 & 0.081 & 0.417 \\
    \hline
    Vertex Barrel & 0.530 & 0.081 & 0.130 & 0.446 & 0.159 & 0.185 & 0.512 & 0.130 & 0.204 & 0.206 & 0.111 & 0.348 \\
    \hline
    Outer Tracker Endcap & 0.531 & 0.226 & 0.161 & 0.150 & 0.117 & 0.346 & 0.528 & 0.234 & 0.163 & 0.165 & 0.084 & 0.354 \

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml

from helpers.data_transforms import load_in_data

plt.style.use("../science.mplstyle")



In [ ]:
with open("configs.yaml", "r") as f:
    configs = yaml.safe_load(f)

ALL_COLLECTIONS = configs["ALL_COLLECTIONS"]
PATH_TO_DATA_DIR = configs["PATH_TO_DATA_DIR"]
NUM_COND_INPUTS = configs["NUM_COND_INPUTS"]
FEATURE_ORDER = configs["FEATURE_ORDER"]

log_vars = []
NUM_BINS = configs["NUM_BINS"]

# Map model directory key -> display label; add/remove entries as needed
MODELS = {
    "tabddpm_rphi": "Diffusion model, global $\phi$",
    "tabddpm_local_phi": "Diffusion model, local $\phi$",
    #   "flow_NCSF_rphi": "Flow model, global $\phi$",
    # "flow_NCSF_local_phi": "Flow model, local $\phi$",
}

PATH_TO_SAVE_SAMPLES = configs["PATH_TO_SAVE_SAMPLES"]


In [ ]:
# load in data

NN = -1

all_data_dir = {}
all_samples_dir = {}  # all_samples_dir[model_key][col_name]
bins_dict = {}

for col_name in ALL_COLLECTIONS:
    X, _ = load_in_data([col_name], "rphi", PATH_TO_DATA_DIR, 1, NUM_COND_INPUTS, FEATURE_ORDER)
    all_data_dir[col_name] = X[:NN]
    bins_dict[col_name] = {}
    for i in range(all_data_dir[col_name].shape[1]):
        if i in log_vars:
            bins_dict[col_name][i] = np.logspace(
                np.log10(0.9 * np.min(all_data_dir[col_name][:, i])),
                np.log10(1.1 * np.max(all_data_dir[col_name][:, i])),
                NUM_BINS,
            )
        else:
            bins_dict[col_name][i] = np.linspace(
                np.min(all_data_dir[col_name][:, i]) - 3,
                np.max(all_data_dir[col_name][:, i]) + 3,
                NUM_BINS,
            )

for model_key in MODELS:
    samples_path = PATH_TO_SAVE_SAMPLES + f"{model_key}/"
    all_samples_dir[model_key] = {}
    for col_name in ALL_COLLECTIONS:
        all_samples_dir[model_key][col_name] = np.load(
            f"{samples_path}/{col_name}.npy", allow_pickle=True
        )[:NN]


In [ ]:
# ML BIB vs Sim BIB: all models overlaid, per collection

from matplotlib.ticker import NullFormatter

n_bins = 80
eps = 1e-12
feature_labels = ["log($E$) [GeV]", "$t$ [s]", "$r$ [mm]", "$\\phi$ [rad]", "$z$ [mm]"]

for col_name in ALL_COLLECTIONS:

    fig, ax = plt.subplots(
        2, 5,
        figsize=(18, 5),
        sharex="col",
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
    )

    for i in range(5):
        data_vals = all_data_dir[col_name][:, i]

        # Common binning across Sim BIB + all models
        bins = np.histogram_bin_edges(data_vals, bins=n_bins)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])

        data_hist, _ = np.histogram(data_vals, bins=bins, density=True)

        # Sim BIB (top panel)
        ax[0, i].hist(
            data_vals, bins=bins, density=True,
            histtype="step", linewidth=1.5, label="Full simulation",
        )
        ax[0, i].set_yscale("log")
        if i == 0:
            ax[0, i].set_ylabel("Density", fontsize=14)
        else:
            ax[0, i].yaxis.set_tick_params(labelleft=False)
            ax[0, i].yaxis.set_ticklabels([])
            ax[0, i].yaxis.set_major_formatter(NullFormatter())
            ax[0, i].yaxis.set_minor_formatter(NullFormatter())
        ax[0, i].tick_params(labelsize=10)

        # Ratio panel baseline
        ax[1, i].axhline(1.0, linestyle="--", linewidth=1, color="k")
        ax[1, i].set_xlabel(feature_labels[i], fontsize=14)
        if i == 0:
            ax[1, i].set_ylabel("Ratio to \nfull simulation", fontsize=12)
        else:
            ax[1, i].yaxis.set_tick_params(labelleft=False)
            ax[1, i].yaxis.set_ticklabels([])
        ax[1, i].tick_params(labelsize=12)

        # Each model overlaid
        for model_key, model_label in MODELS.items():
            flow_vals = all_samples_dir[model_key][col_name][:, i]
            flow_hist, _ = np.histogram(flow_vals, bins=bins, density=True)
            ratio = np.divide(
                flow_hist, data_hist,
                out=np.full_like(flow_hist, np.nan, dtype=float),
                where=data_hist > eps,
            )

            ax[0, i].hist(
                flow_vals, bins=bins, density=True,
                histtype="step", linewidth=1.5, label=model_label,
            )
            ax[1, i].step(bin_centers, ratio, where="mid", linewidth=1.5, label=model_label)

    handles, labels = ax[0, 0].get_legend_handles_labels()
    ax[0, -1].legend(
        handles, labels,
        loc="lower center" if "Barrel" in col_name else "upper center",
        #bbox_to_anchor=(0.7, 1.0),
        ncol=1,
        frameon=False,
        fontsize=12,
    )

    #fig.suptitle(col_name, fontsize=14, y=0.95)
    plt.subplots_adjust(wspace=0.05)
    plt.savefig(f"plots/diff_vs_sim_{col_name}.pdf", bbox_inches="tight", dpi=300)
    plt.show()


In [ ]:
# # Sim BIB

# feature_labels = ["log($E$) [GeV]", "$t$ [s]", "$r$ [mm]", "$\\phi$", "$z$ [mm]"]
# from matplotlib.ticker import NullFormatter

# fig, ax = plt.subplots(
#     1, 5,
#     figsize=(22, 4),
#     sharey=False
# )

# for i in range(5):

#     # Common binning across collections
#     vals = np.concatenate([
#         all_data_dir[col_name][:, i]
#         for col_name in ALL_COLLECTIONS
#     ])
#     bins = np.histogram_bin_edges(vals, bins=50)

#     for col_name in ALL_COLLECTIONS:
#         ax[i].hist(
#             all_data_dir[col_name][:, i],
#             bins=bins,
#             density=True,
#             histtype="step",
#             linewidth=1.5,
#             label=col_name,
#         )

#     ax[i].set_xlabel(feature_labels[i], fontsize=14)

#     ax[i].set_yscale("log")

#     # Only left-most panel gets y label
#     if i == 0:
#         ax[i].set_ylabel("Density", fontsize=14)
#     else:
#         ax[i].yaxis.set_tick_params(labelleft=False)
#         ax[i].yaxis.set_ticklabels([])  # explicitly clear tick labels
#         ax[i].yaxis.set_major_formatter(NullFormatter())
#         ax[i].yaxis.set_minor_formatter(NullFormatter())

#     ax[i].tick_params(labelsize=12)
    

# # Single legend above all panels
# handles, labels = ax[0].get_legend_handles_labels()
# fig.legend(
#     handles,
#     labels,
#     loc="upper center",
#     bbox_to_anchor=(0.65, 1.05),
#     ncol=3,
#     frameon=False,
#     fontsize=14,
# )

# # Reduce spacing between panels
# plt.subplots_adjust(
#     # left=0.06,
#     # right=0.995,
#     # bottom=0.18,
#     # top=0.78,
#     wspace=0.05,
# )

# plt.show()





In [ ]:

# plt.figure()
# for col_name in ALL_COLLECTIONS:
#     plt.scatter(np.abs(all_data_dir[col_name][:1000000,4]), all_data_dir[col_name][:1000000,2], label = col_name, s = 0.00001)

# plt.ylabel("$r$ [mm]", fontsize = 14)
# plt.xlabel("$|z|$ [mm]", fontsize = 14)
# plt.gca().set_aspect("equal", adjustable="box")

# plt.legend(markerscale=1000, loc = (1,0), fontsize = 14)
# plt.show()


In [ ]:
plt.figure()
plt.hist(
    np.abs(all_data_dir["VertexEndcapCollection"][:, 4]),
    bins=np.linspace(75, 100, 1000),
    density=True,
    histtype="step",
    linewidth=1.5,
    label="OuterTrackerEndcapCollection",
)
plt.show()

# 7
